In [ ]:
# ============================================================
# CONFIGURATION - every path comes from config/paths.py, the single
# source of truth. Override cluster locations with the MUSICA_ENV_*
# environment variables documented there. Do not hard-code paths here.
# ============================================================
import sys, pathlib
_here = pathlib.Path.cwd().resolve()
_ROOT = next(p for p in [_here, *_here.parents]
             if (p / 'config' / 'paths.py').exists())
sys.path.insert(0, str(_ROOT))
import config  # also puts functions/ on sys.path
from config import paths as P


In [ ]:
figure_diri = f'{P.FIGURES_ROOT}/CESM_analysis/BGO3/'

### Download figures to local disk: 
# rsync -avz --exclude=".*" -e ssh "<username>@svante9.mit.edu:{P.FIGURES_ROOT}/CESM_analysis/BGO3/*" "/Users/$USER/Downloads/"
# rsync -avz --exclude=".*" -e ssh "<username>@svante9.mit.edu:{P.FIGURES_ROOT}/CESM_analysis/BGO3/*" "/Users/$USER/Downloads/同步空间/MUSICA(CESM22-SE)/Y2022_BackgroundO3/Figures/"

### Functions

In [ ]:
import os
import glob
import fnmatch

import pandas as pd
import geopandas as gpd
from shapely import wkt
from shapely.geometry import Point, Polygon

import xarray as xr
import numpy as np

import matplotlib.pyplot as plt # Core library for plotting
import matplotlib.cm as cm # To use different colormaps
import cartopy.crs as ccrs # For map projection
import seaborn as sns # boxplot

from matplotlib.cm import ScalarMappable


In [ ]:
import sys
# functions I defined
sys.path.insert(0,f'{P.HOME_ROOT}/Scripts/CESM_analysis/functions/')
from Plot_2D import Plot_2D # To draw a map
from func_MUSICA_DefineRegion import *

SCRIP_ne30 = f'{P.HOME_ROOT}/Scripts/CESM_analysis/functions/ne30np4_091226_pentagons.nc'

In [ ]:
def extract_date_correctly(filename):
    # Extracting the date portion from filename based on observed structure
    date_section = filename.split('.')[-2].split('-')
    # Combine the first three elements to form the date in YYYY-MM-DD
    full_date = '-'.join(date_section[:3])
    return full_date


In [ ]:
# Specify timezone for the given region
def regional_UTCtimezone_offset_summer(fileregion):
    """This function returns the timezone of a given region
       *With daylight saving  
       Options: WestCoast,Mountain,Midwest,Southwest,Southeast,Northeast
    """
    # Define regions        
    if fileregion == "WestCoast":
        # Pacific (UTC-7)
        UTCtimezone_offset = -7
    elif fileregion == "Mountain":
        # Mountain (UTC-6)
        UTCtimezone_offset = -6
    elif fileregion == "Midwest":
        # Central (UTC-5)
        UTCtimezone_offset = -5
    elif fileregion == "Southwest":
        # Central (UTC-5)
        UTCtimezone_offset = -5
    elif fileregion == "Southeast":
        # Eastern (UTC-4)
        UTCtimezone_offset = -4
    elif fileregion == "Northeast":
        # Eastern (UTC-4)
        UTCtimezone_offset = -4
    
    return UTCtimezone_offset

In [ ]:
varlabel_dic = {'O3':r'$O_{3}$',
                'NO2':r'$NO_{2}$',
                'Ox':r'$O_{x}$',
                'NOx':r'$NO_{x}$',
                'HCHO':r'$HCHO$', # listing twice if pointed
               'CH2O':r'$HCHO$', #r'$CH_{2}O$'
                'CO':r'$CO$',
                'SO2':r'$SO_{2}$',
                'PM25':r'$PM_{2.5}$',
               }

unit_dic = {'O3':r'$ppb$',
                'NO2':r'$ppb$',
            'Ox':r'$ppb$',
                'NOx':r'$ppb$',
            'HCHO':r'$ppb$', # listing twice if pointed
               'CH2O':r'$ppb$',
                'CO':r'$ppb$',
                'SO2':r'$ppb$',
                'PM25':r'$\mu g/m^{3}$',   #r'$kg/m^{3}$', 
               }

Scalefactor_dic = {'O3':1e9,
                'NO2':1e9,
                'Ox':1e9,
                'NOx':1e9,
               'CH2O':1e9,
                'CO':1e9,
                'SO2':1e9,
                'PM25':1e9,
               }

In [ ]:
PerturbedAEmis = [
    'NO','NH3','CO',
    'C2H2','C2H4','C2H5OH','C2H6','C3H6','C3H8','CH2O',
    'CH3CHO','CH3COCH3','CH3OH', 
    
    'ISOP','MEK','MTERP',
    'BENZENE','BIGENE','TOLUENE',
    'SVOC',
    'SO2',
    'bc_a4','num_bc_a4','num_pom_a4','pom_a4'
]

In [ ]:
region_bounds = {
        "CONUS": {"lon_range": [-140, -50], "lat_range": [15, 60]},
        "Global": {"lon_range": None, "lat_range": None},
        "NYC": {"lon_range": [-74.5, -73.5], "lat_range": [40.4, 41]},
        "ExtendedNYC": {"lon_range": [-75, -72], "lat_range": [40, 42]}
    }

In [ ]:
Abs_rangeMax_dic = {'NO':7e-11,
                  'CO':5e-10,
                  'SO2':3e-11, 
                  'bc_a4':3e-12,
                  'NH3':2e-11,
                  'C2H2':2e-12, 
                  'C2H4':5e-12, # ethene
                  'C2H5OH':5e-11,
                  'C2H6':6e-12, # ethane
                  'C3H6':2e-11, 
                  'C3H8':2e-12, 
                  'CH3OH':2e-12,
                  'CH3CHO':2e-12, # 
                  'CH3COCH3':2e-12, 
                  'MEK':2e-12,
                    
                # Second panel
                  'BENZENE':3e-12, 
                  'TOLUENE':7e-12,
                  'BIGENE':4e-12,   
                  'MTERP':5e-13, 
                  'ISOP':8e-14, 
                  'CH2O':3e-12, # formaldehyde
                  'SVOC':6e-12,
                   }

## Note that the magnitude for some VOC species is much lower compared to their biogenic emissions
# MTERP: 4.90e-13 | 2.14e-13 (anthrop) compared to 
# ISOP: 8.37e-14 | 7.34e-14 (anthrop) compared to 

import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'DejaVu Sans'  # Change to a font that supports the superscript characters

def get_superscript(magnitude):
    return r'$10^{' + str(magnitude) + '}$'

In [ ]:
def molecules_to_kg_per_m2_per_s(molecules_per_cm2_per_s, molecular_weight):
    """
        Read in emissions data in [molecules cm-2 s-1], and molecular_weight in [g/mole]
        Return emissions in [kg m-2 s-1]    
    """
    # Constants
    avogadro_number = 6.022e23  # molecules per mole
    m2_to_cm2 = 1e4  # square meters to square centimeters 
    kg_to_g = 1e3

    # Convert 
    kg_per_m2_per_s = molecules_per_cm2_per_s*(1/avogadro_number)*molecular_weight*(1/kg_to_g)*(m2_to_cm2)

    return kg_per_m2_per_s

# # Example usage
# molecules_per_cm2_per_s = 1  # for example, 1e18 molecules per cm^2 per second
# molecular_weight = 30  # molecular weight of water in g/mol

# result = molecules_to_kg_per_m2_per_s(molecules_per_cm2_per_s, molecular_weight)
# print("Result:", result, "kg/m^2/s")


setunit = r'$kg$ $m^{-2}s^{-1}$'

## Fire Emissions

In [ ]:
### Read in original emissions data
original_diri = f'{P.NCAR_COPIES_ROOT}/acom/MUSICA/emissions/qfed2.6_finn/ne30np4/'
CONUSlandMasked_diri = f'{P.NCAR_COPIES_ROOT}/acom/MUSICA/emissions/qfed2.6_finn/ne30np4_CONUSlandMasked_80kmBuffer/'

In [ ]:
# Search for files that match the pattern
import glob
import os

# Directory where files are stored
# CAMS_diri = '/path/to/your/files'  # Make sure this is correctly defined

# Updated pattern with wildcard
pattern = os.path.join(original_diri, 'qfed.emis_*_bb_surface_daily_20171201T20231231_ne30np4_mol_c20240126.nc')

# Get list of matching files in full path
file_list = glob.glob(pattern)
# print(CAMS_v51_file_list)

# Extract only the filenames
file_names = [os.path.basename(f) for f in file_list]

# print(file_names[:3])

spc_ls = []
for spcIdx in range(len(file_list)):
    spc_name = file_names[spcIdx].split('_')[1]
    spc_ls.append(spc_name)
    
import re
# Extract species name between 'ne30np4_' and '_c20210423'
species_names = [
    re.search(r'qfed.emis_(.+?)_bb_surface_daily_20171201T20231231_ne30np4_mol_c20240126.nc', f).group(1)
    for f in file_names
]
species_names = sorted(species_names)
print(species_names)

In [ ]:
## Test with one species
spc = 'CO'
spc_fileINpath = f'{original_diri}qfed.emis_{spc}_bb_surface_daily_20171201T20231231_ne30np4_mol_c20240126.nc'
# Open Xarray dataset
spci_ds = xr.open_dataset(spc_fileINpath)
spci_ds

In [ ]:
import xarray as xr
import pandas as pd

# Specify the year and month of interest
Yeari = '2022'
Monthi = '07'

# Convert to integers
year_int = int(Yeari)
month_int = int(Monthi)

# Select the desired month directly using .dt.year and .dt.month
ds_monthi = spci_ds.sel(time=(spci_ds.time.dt.year == int(Yeari)) & (spci_ds.time.dt.month == int(Monthi)))

# Compute the monthly mean of the 'emiss' variable
emiss_monthi_mean = ds_monthi['emiss'].mean(dim='time')


In [ ]:
emiss_monthi_mean

In [ ]:
# Plot emissions monthly means | 2023
lev_idx = -1

Yeari = '2022'
Months_ls = ['04','05','06','07','08','09','10']

# Map setting
setcmap = cm.CMRmap_r
setunit_size, settitle_size, settitle_size2, setcolortick_size = [20, 25, 49, 35]

### Original fire emissions for monthly mean
spc = 'CO'
givenMW = 28 #molecular weight in g/mol
spc_fileINpath = f'{original_diri}qfed.emis_{spc}_bb_surface_daily_20171201T20231231_ne30np4_mol_c20240126.nc'
# Open Xarray dataset
spci_ds = xr.open_dataset(spc_fileINpath)

###-----------------------------------
### Plot a x rows by x columns map
nrows, ncols = 2, 4

# Create a new figure with subplots sharing x and y axes
fig, axes = plt.subplots(
    nrows=nrows, 
    ncols=ncols, 
    figsize=(5.9 * ncols, 3.64 * nrows), 
    sharex=True, 
    sharey=True, 
    subplot_kw={'projection': ccrs.PlateCarree()},  # Set projection for all subplots
    gridspec_kw={'wspace': 0, 'hspace': 0.15}  # Remove space between subplots
)
# fig.tight_layout(pad=0)  # Remove padding

# Quick plot to check
setmap = cm.CMRmap_r 

for subploti in range(len(Months_ls)):
    ax = axes.flat[subploti]
    # ax.set_extent([lon_range[0], lon_range[1], lat_range[0], lat_range[1]], crs=ccrs.PlateCarree())
    if subploti<=6:
        # Get the corresponding subplot
        # Get the data in the corresponding month
        Monthi = Months_ls[subploti]
        Timei = f'{Yeari}T{Monthi}'
        
        """Get the data for the given month (in ne30)"""
        # Convert to integers
        year_int = int(Yeari)
        month_int = int(Monthi)
        # Select the desired month directly using .dt.year and .dt.month
        ds_monthi = spci_ds.sel(time=(spci_ds.time.dt.year == int(Yeari)) & (spci_ds.time.dt.month == int(Monthi)))

        # Compute the monthly mean of the 'emiss' variable
        emiss_monthi_mean = ds_monthi['emiss'].mean(dim='time')

        Plot_ar = emiss_monthi_mean.values#*scalefactor

        vari_kg_per_m2_per_s = molecules_to_kg_per_m2_per_s(Plot_ar, givenMW)
        Plot_unit = r'$kg$ $m^{-2}$ $s^{-1}$'
        longname = spci_ds['emiss'].long_name
        # rangeMax = np.nanmean(Plot_ar) #setvmax*scalefactor
        
        # get rangeMax
        rangeMax = Abs_rangeMax_dic[spc]

        ### Accounting for scientific notation
        # Get the magnitude in scientific notation, adjust for e-
        magnitude = int(np.floor(np.log10(abs(rangeMax))))
        # Format the magnitude as a superscript
        formatted_magnitude = get_superscript(str(magnitude))
        modified_rangeMax = rangeMax*(1/(10)**magnitude)

        setunit = f'{formatted_magnitude}{Plot_unit}'
        
        # show colorbar for the last row in each column 
        im = Plot_2D( vari_kg_per_m2_per_s*(1/(10)**magnitude), scrip_file=SCRIP_ne30,  ax=ax,
                    unit=setunit,unit_size=setunit_size,colortick_size=setcolortick_size,
                    cmin=0.1, cmax=modified_rangeMax,
                     state=True, 
                     # lon_range=[lon_right, lon_left], lat_range=[lat_bot, lat_up], 
                     lon_range=[-140,-50], lat_range=[15,60],
                     cmap=setcmap, colorbar=False, 
                    grid_line=False, onedecimal=True)
        
        
        # # add title label
        settitle = f'{Timei} Mean'
        # settitle = f'{caseDiffi_label}'
        ax.set_title(f'{settitle}',c='k',fontsize=settitle_size,y=1.02, fontweight='bold'
                    # bbox=dict(facecolor='white', edgecolor='none', boxstyle='round,pad=0.01'),y=0.87,
                    )        
        # If this is the last column, add a colorbar
        # if subploti == 3:
        if subploti == 6:
            norm = plt.Normalize(vmin=0.1, vmax=modified_rangeMax)
            # Create a dummy color-mappable object for the colorbar
            dummy_mappable = ScalarMappable(norm=norm, cmap=setcmap)
            dummy_mappable.set_array([])  # Required for colorbar creation

            # fig.add_axes([left, bottom, width, height])
            cax = fig.add_axes([ax.get_position().x1+0.002, ax.get_position().y0, 0.01, ax.get_position().height*1])
            cbar = fig.colorbar(dummy_mappable, cax=cax, orientation='vertical', extend='both') #extend='both'
            cbar.set_label(setunit, fontsize=setunit_size)
            cbar.ax.tick_params(labelsize=setcolortick_size)
        
        # Hide x and y-axis ticks and labels
        ax.set_xticks([])
        ax.set_yticks([])
        
    elif subploti>6:
        # ax.axis('off')
        # ax.set_axis_off()
        ax.set_visible(False) 
        
axes.flat[-1].set_visible(False)        

# Overall title
plttitle = f'BB Emissions | {spc}'
fig.suptitle(plttitle, fontsize=28, y=1.02, fontweight='bold')

savefig_filename = f'{figure_diri}BBemis_202204T10_CONUS.png'
# Save the figure
plt.savefig(savefig_filename, dpi=300, bbox_inches='tight')  # Adjust the filename and dpi as needed
print(f'Save to {savefig_filename}')
plt.show()

In [ ]:
# Plot emissions monthly means | CONUSlandMasked_diri
lev_idx = -1

Yeari = '2022'
Months_ls = ['04','05','06','07','08','09','10']

# Map setting
setcmap = cm.CMRmap_r
setunit_size, settitle_size, settitle_size2, setcolortick_size = [20, 25, 49, 35]

### Original fire emissions for monthly mean
spc = 'CO'
givenMW = 28 #molecular weight in g/mol
spc_fileINpath = f'{CONUSlandMasked_diri}qfed.emis_{spc}_bb_surface_daily_20171201T20231231_ne30np4_mol_c20240126.nc'
# Open Xarray dataset
spci_ds = xr.open_dataset(spc_fileINpath)

###-----------------------------------
### Plot a x rows by x columns map
nrows, ncols = 2, 4

# Create a new figure with subplots sharing x and y axes
fig, axes = plt.subplots(
    nrows=nrows, 
    ncols=ncols, 
    figsize=(5.9 * ncols, 3.64 * nrows), 
    sharex=True, 
    sharey=True, 
    subplot_kw={'projection': ccrs.PlateCarree()},  # Set projection for all subplots
    gridspec_kw={'wspace': 0, 'hspace': 0.15}  # Remove space between subplots
)
# fig.tight_layout(pad=0)  # Remove padding

# Quick plot to check
setmap = cm.CMRmap_r 

for subploti in range(len(Months_ls)):
    ax = axes.flat[subploti]
    # ax.set_extent([lon_range[0], lon_range[1], lat_range[0], lat_range[1]], crs=ccrs.PlateCarree())
    if subploti<=6:
        # Get the corresponding subplot
        # Get the data in the corresponding month
        Monthi = Months_ls[subploti]
        Timei = f'{Yeari}T{Monthi}'
        
        """Get the data for the given month (in ne30)"""
        # Convert to integers
        year_int = int(Yeari)
        month_int = int(Monthi)
        # Select the desired month directly using .dt.year and .dt.month
        ds_monthi = spci_ds.sel(time=(spci_ds.time.dt.year == int(Yeari)) & (spci_ds.time.dt.month == int(Monthi)))

        # Compute the monthly mean of the 'emiss' variable
        emiss_monthi_mean = ds_monthi['emiss'].mean(dim='time')

        Plot_ar = emiss_monthi_mean.values#*scalefactor

        vari_kg_per_m2_per_s = molecules_to_kg_per_m2_per_s(Plot_ar, givenMW)
        Plot_unit = r'$kg$ $m^{-2}$ $s^{-1}$'
        longname = spci_ds['emiss'].long_name
        # rangeMax = np.nanmean(Plot_ar) #setvmax*scalefactor
        
        # get rangeMax
        rangeMax = Abs_rangeMax_dic[spc]

        ### Accounting for scientific notation
        # Get the magnitude in scientific notation, adjust for e-
        magnitude = int(np.floor(np.log10(abs(rangeMax))))
        # Format the magnitude as a superscript
        formatted_magnitude = get_superscript(str(magnitude))
        modified_rangeMax = rangeMax*(1/(10)**magnitude)

        setunit = f'{formatted_magnitude}{Plot_unit}'
        
        # show colorbar for the last row in each column 
        im = Plot_2D( vari_kg_per_m2_per_s*(1/(10)**magnitude), scrip_file=SCRIP_ne30,  ax=ax,
                    unit=setunit,unit_size=setunit_size,colortick_size=setcolortick_size,
                    cmin=0.1, cmax=modified_rangeMax,
                     state=True, 
                     # lon_range=[lon_right, lon_left], lat_range=[lat_bot, lat_up], 
                     lon_range=[-140,-50], lat_range=[15,60],
                     cmap=setcmap, colorbar=False, 
                    grid_line=False, onedecimal=True)
        
        
        # # add title label
        settitle = f'{Timei} Mean'
        # settitle = f'{caseDiffi_label}'
        ax.set_title(f'{settitle}',c='k',fontsize=settitle_size,y=1.02, fontweight='bold'
                    # bbox=dict(facecolor='white', edgecolor='none', boxstyle='round,pad=0.01'),y=0.87,
                    )        
        # If this is the last column, add a colorbar
        # if subploti == 3:
        if subploti == 6:
            norm = plt.Normalize(vmin=0.1, vmax=modified_rangeMax)
            # Create a dummy color-mappable object for the colorbar
            dummy_mappable = ScalarMappable(norm=norm, cmap=setcmap)
            dummy_mappable.set_array([])  # Required for colorbar creation

            # fig.add_axes([left, bottom, width, height])
            cax = fig.add_axes([ax.get_position().x1+0.002, ax.get_position().y0, 0.01, ax.get_position().height*1])
            cbar = fig.colorbar(dummy_mappable, cax=cax, orientation='vertical', extend='both') #extend='both'
            cbar.set_label(setunit, fontsize=setunit_size)
            cbar.ax.tick_params(labelsize=setcolortick_size)
        
        # Hide x and y-axis ticks and labels
        ax.set_xticks([])
        ax.set_yticks([])
        
    elif subploti>6:
        # ax.axis('off')
        # ax.set_axis_off()
        ax.set_visible(False) 
        
axes.flat[-1].set_visible(False)        

# Overall title
plttitle = f'BB Emissions | {spc}'
fig.suptitle(plttitle, fontsize=28, y=1.02, fontweight='bold')

savefig_filename = f'{figure_diri}noCONUSMasked_BBemis_202204T10_CONUS.png'
# Save the figure
plt.savefig(savefig_filename, dpi=300, bbox_inches='tight')  # Adjust the filename and dpi as needed
print(f'Save to {savefig_filename}')
plt.show()

In [ ]:
# Plot emissions monthly means | CONUSlandMasked_diri
lev_idx = -1

Yeari = '2023'
Months_ls = ['04','05','06','07','08','09','10']

# Map setting
setcmap = cm.CMRmap_r
setunit_size, settitle_size, settitle_size2, setcolortick_size = [20, 25, 49, 35]

### Original fire emissions for monthly mean
spc = 'CO'
givenMW = 28 #molecular weight in g/mol
spc_fileINpath = f'{CONUSlandMasked_diri}qfed.emis_{spc}_bb_surface_daily_20171201T20231231_ne30np4_mol_c20240126.nc'
# Open Xarray dataset
spci_ds = xr.open_dataset(spc_fileINpath)

###-----------------------------------
### Plot a x rows by x columns map
nrows, ncols = 2, 4

# Create a new figure with subplots sharing x and y axes
fig, axes = plt.subplots(
    nrows=nrows, 
    ncols=ncols, 
    figsize=(5.9 * ncols, 3.64 * nrows), 
    sharex=True, 
    sharey=True, 
    subplot_kw={'projection': ccrs.PlateCarree()},  # Set projection for all subplots
    gridspec_kw={'wspace': 0, 'hspace': 0.15}  # Remove space between subplots
)
# fig.tight_layout(pad=0)  # Remove padding

# Quick plot to check
setmap = cm.CMRmap_r 

for subploti in range(len(Months_ls)):
    ax = axes.flat[subploti]
    # ax.set_extent([lon_range[0], lon_range[1], lat_range[0], lat_range[1]], crs=ccrs.PlateCarree())
    if subploti<=6:
        # Get the corresponding subplot
        # Get the data in the corresponding month
        Monthi = Months_ls[subploti]
        Timei = f'{Yeari}T{Monthi}'
        
        """Get the data for the given month (in ne30)"""
        # Convert to integers
        year_int = int(Yeari)
        month_int = int(Monthi)
        # Select the desired month directly using .dt.year and .dt.month
        ds_monthi = spci_ds.sel(time=(spci_ds.time.dt.year == int(Yeari)) & (spci_ds.time.dt.month == int(Monthi)))

        # Compute the monthly mean of the 'emiss' variable
        emiss_monthi_mean = ds_monthi['emiss'].mean(dim='time')

        Plot_ar = emiss_monthi_mean.values#*scalefactor

        vari_kg_per_m2_per_s = molecules_to_kg_per_m2_per_s(Plot_ar, givenMW)
        Plot_unit = r'$kg$ $m^{-2}$ $s^{-1}$'
        longname = spci_ds['emiss'].long_name
        # rangeMax = np.nanmean(Plot_ar) #setvmax*scalefactor
        
        # get rangeMax
        rangeMax = Abs_rangeMax_dic[spc]

        ### Accounting for scientific notation
        # Get the magnitude in scientific notation, adjust for e-
        magnitude = int(np.floor(np.log10(abs(rangeMax))))
        # Format the magnitude as a superscript
        formatted_magnitude = get_superscript(str(magnitude))
        modified_rangeMax = rangeMax*(1/(10)**magnitude)

        setunit = f'{formatted_magnitude}{Plot_unit}'
        
        # show colorbar for the last row in each column 
        im = Plot_2D( vari_kg_per_m2_per_s*(1/(10)**magnitude), scrip_file=SCRIP_ne30,  ax=ax,
                    unit=setunit,unit_size=setunit_size,colortick_size=setcolortick_size,
                    cmin=0.1, cmax=modified_rangeMax,
                     state=True, 
                     # lon_range=[lon_right, lon_left], lat_range=[lat_bot, lat_up], 
                     lon_range=[-140,-50], lat_range=[15,60],
                     cmap=setcmap, colorbar=False, 
                    grid_line=False, onedecimal=True)
        
        
        # # add title label
        settitle = f'{Timei} Mean'
        # settitle = f'{caseDiffi_label}'
        ax.set_title(f'{settitle}',c='k',fontsize=settitle_size,y=1.02, fontweight='bold'
                    # bbox=dict(facecolor='white', edgecolor='none', boxstyle='round,pad=0.01'),y=0.87,
                    )        
        # If this is the last column, add a colorbar
        # if subploti == 3:
        if subploti == 6:
            norm = plt.Normalize(vmin=0.1, vmax=modified_rangeMax)
            # Create a dummy color-mappable object for the colorbar
            dummy_mappable = ScalarMappable(norm=norm, cmap=setcmap)
            dummy_mappable.set_array([])  # Required for colorbar creation

            # fig.add_axes([left, bottom, width, height])
            cax = fig.add_axes([ax.get_position().x1+0.002, ax.get_position().y0, 0.01, ax.get_position().height*1])
            cbar = fig.colorbar(dummy_mappable, cax=cax, orientation='vertical', extend='both') #extend='both'
            cbar.set_label(setunit, fontsize=setunit_size)
            cbar.ax.tick_params(labelsize=setcolortick_size)
        
        # Hide x and y-axis ticks and labels
        ax.set_xticks([])
        ax.set_yticks([])
        
    elif subploti>6:
        # ax.axis('off')
        # ax.set_axis_off()
        ax.set_visible(False) 
        
axes.flat[-1].set_visible(False)        

# Overall title
plttitle = f'BB Emissions | {spc}'
fig.suptitle(plttitle, fontsize=28, y=1.02, fontweight='bold')

savefig_filename = f'{figure_diri}noCONUSMasked_BBemis_202304T10_CONUS.png'
# Save the figure
plt.savefig(savefig_filename, dpi=300, bbox_inches='tight')  # Adjust the filename and dpi as needed
print(f'Save to {savefig_filename}')
plt.show()

In [ ]:
# Plot emissions monthly means | 2023
lev_idx = -1

Yeari = '2022'
Months_ls = ['04','05','06','07','08','09','10']

# Map setting
setcmap = cm.CMRmap_r
setunit_size, settitle_size, settitle_size2, setcolortick_size = [20, 25, 49, 35]

### Original fire emissions for monthly mean
spc = 'CO'
givenMW = 28 #molecular weight in g/mol
spc_fileINpath = f'{original_diri}qfed.emis_{spc}_bb_surface_daily_20171201T20231231_ne30np4_mol_c20240126.nc'
# Open Xarray dataset
spci_ds = xr.open_dataset(spc_fileINpath)

###-----------------------------------
### Plot a x rows by x columns map
nrows, ncols = 2, 4

# Create a new figure with subplots sharing x and y axes
fig, axes = plt.subplots(
    nrows=nrows, 
    ncols=ncols, 
    figsize=(5.9 * ncols, 3.64 * nrows), 
    sharex=True, 
    sharey=True, 
    subplot_kw={'projection': ccrs.PlateCarree()},  # Set projection for all subplots
    gridspec_kw={'wspace': 0, 'hspace': 0.15}  # Remove space between subplots
)
# fig.tight_layout(pad=0)  # Remove padding

# Quick plot to check
setmap = cm.CMRmap_r 

for subploti in range(len(Months_ls)):
    ax = axes.flat[subploti]
    # ax.set_extent([lon_range[0], lon_range[1], lat_range[0], lat_range[1]], crs=ccrs.PlateCarree())
    if subploti<=6:
        # Get the corresponding subplot
        # Get the data in the corresponding month
        Monthi = Months_ls[subploti]
        Timei = f'{Yeari}T{Monthi}'
        
        """Get the data for the given month (in ne30)"""
        # Convert to integers
        year_int = int(Yeari)
        month_int = int(Monthi)
        # Select the desired month directly using .dt.year and .dt.month
        ds_monthi = spci_ds.sel(time=(spci_ds.time.dt.year == int(Yeari)) & (spci_ds.time.dt.month == int(Monthi)))

        # Compute the monthly mean of the 'emiss' variable
        emiss_monthi_mean = ds_monthi['emiss'].mean(dim='time')

        Plot_ar = emiss_monthi_mean.values#*scalefactor

        vari_kg_per_m2_per_s = molecules_to_kg_per_m2_per_s(Plot_ar, givenMW)
        Plot_unit = r'$kg$ $m^{-2}$ $s^{-1}$'
        longname = spci_ds['emiss'].long_name
        # rangeMax = np.nanmean(Plot_ar) #setvmax*scalefactor
        
        # get rangeMax
        rangeMax = Abs_rangeMax_dic[spc]

        ### Accounting for scientific notation
        # Get the magnitude in scientific notation, adjust for e-
        magnitude = int(np.floor(np.log10(abs(rangeMax))))
        # Format the magnitude as a superscript
        formatted_magnitude = get_superscript(str(magnitude))
        modified_rangeMax = rangeMax*(1/(10)**magnitude)

        setunit = f'{formatted_magnitude}{Plot_unit}'
        
        # show colorbar for the last row in each column 
        im = Plot_2D( vari_kg_per_m2_per_s*(1/(10)**magnitude), scrip_file=SCRIP_ne30,  ax=ax,
                    unit=setunit,unit_size=setunit_size,colortick_size=setcolortick_size,
                    cmin=0.1, cmax=modified_rangeMax,
                     cmap=setcmap, colorbar=False, 
                    grid_line=False, onedecimal=True)
        
        
        # # add title label
        settitle = f'{Timei} Mean'
        # settitle = f'{caseDiffi_label}'
        ax.set_title(f'{settitle}',c='k',fontsize=settitle_size,y=1.02, fontweight='bold'
                    # bbox=dict(facecolor='white', edgecolor='none', boxstyle='round,pad=0.01'),y=0.87,
                    )        
        # If this is the last column, add a colorbar
        # if subploti == 3:
        if subploti == 6:
            norm = plt.Normalize(vmin=0.1, vmax=modified_rangeMax)
            # Create a dummy color-mappable object for the colorbar
            dummy_mappable = ScalarMappable(norm=norm, cmap=setcmap)
            dummy_mappable.set_array([])  # Required for colorbar creation

            # fig.add_axes([left, bottom, width, height])
            cax = fig.add_axes([ax.get_position().x1+0.002, ax.get_position().y0, 0.01, ax.get_position().height*1])
            cbar = fig.colorbar(dummy_mappable, cax=cax, orientation='vertical', extend='both') #extend='both'
            cbar.set_label(setunit, fontsize=setunit_size)
            cbar.ax.tick_params(labelsize=setcolortick_size)
        
        # Hide x and y-axis ticks and labels
        ax.set_xticks([])
        ax.set_yticks([])
        
    elif subploti>6:
        # ax.axis('off')
        # ax.set_axis_off()
        ax.set_visible(False) 
        
axes.flat[-1].set_visible(False)        

# Overall title
plttitle = f'BB Emissions | {spc}'
fig.suptitle(plttitle, fontsize=28, y=1.02, fontweight='bold')

savefig_filename = f'{figure_diri}BBemis_202204T10_Global.png'
# Save the figure
plt.savefig(savefig_filename, dpi=300, bbox_inches='tight')  # Adjust the filename and dpi as needed
print(f'Save to {savefig_filename}')
plt.show()

In [ ]:
# Plot emissions monthly means | 2023
lev_idx = -1

Yeari = '2023'
Months_ls = ['04','05','06','07','08','09','10']

# Map setting
setcmap = cm.CMRmap_r
setunit_size, settitle_size, settitle_size2, setcolortick_size = [20, 25, 49, 35]

### Original fire emissions for monthly mean
spc = 'CO'
givenMW = 28 #molecular weight in g/mol
spc_fileINpath = f'{original_diri}qfed.emis_{spc}_bb_surface_daily_20171201T20231231_ne30np4_mol_c20240126.nc'
# Open Xarray dataset
spci_ds = xr.open_dataset(spc_fileINpath)

###-----------------------------------
### Plot a x rows by x columns map
nrows, ncols = 2, 4

# Create a new figure with subplots sharing x and y axes
fig, axes = plt.subplots(
    nrows=nrows, 
    ncols=ncols, 
    figsize=(5.9 * ncols, 3.64 * nrows), 
    sharex=True, 
    sharey=True, 
    subplot_kw={'projection': ccrs.PlateCarree()},  # Set projection for all subplots
    gridspec_kw={'wspace': 0, 'hspace': 0.15}  # Remove space between subplots
)
# fig.tight_layout(pad=0)  # Remove padding

# Quick plot to check
setmap = cm.CMRmap_r 

for subploti in range(len(Months_ls)):
    ax = axes.flat[subploti]
    # ax.set_extent([lon_range[0], lon_range[1], lat_range[0], lat_range[1]], crs=ccrs.PlateCarree())
    if subploti<=6:
        # Get the corresponding subplot
        # Get the data in the corresponding month
        Monthi = Months_ls[subploti]
        Timei = f'{Yeari}T{Monthi}'
        
        """Get the data for the given month (in ne30)"""
        # Convert to integers
        year_int = int(Yeari)
        month_int = int(Monthi)
        # Select the desired month directly using .dt.year and .dt.month
        ds_monthi = spci_ds.sel(time=(spci_ds.time.dt.year == int(Yeari)) & (spci_ds.time.dt.month == int(Monthi)))

        # Compute the monthly mean of the 'emiss' variable
        emiss_monthi_mean = ds_monthi['emiss'].mean(dim='time')

        Plot_ar = emiss_monthi_mean.values#*scalefactor

        vari_kg_per_m2_per_s = molecules_to_kg_per_m2_per_s(Plot_ar, givenMW)
        Plot_unit = r'$kg$ $m^{-2}$ $s^{-1}$'
        longname = spci_ds['emiss'].long_name
        # rangeMax = np.nanmean(Plot_ar) #setvmax*scalefactor
        
        # get rangeMax
        rangeMax = Abs_rangeMax_dic[spc]

        ### Accounting for scientific notation
        # Get the magnitude in scientific notation, adjust for e-
        magnitude = int(np.floor(np.log10(abs(rangeMax))))
        # Format the magnitude as a superscript
        formatted_magnitude = get_superscript(str(magnitude))
        modified_rangeMax = rangeMax*(1/(10)**magnitude)

        setunit = f'{formatted_magnitude}{Plot_unit}'
        
        # show colorbar for the last row in each column 
        im = Plot_2D( vari_kg_per_m2_per_s*(1/(10)**magnitude), scrip_file=SCRIP_ne30,  ax=ax,
                    unit=setunit,unit_size=setunit_size,colortick_size=setcolortick_size,
                    cmin=0.1, cmax=modified_rangeMax,
                     state=True, 
                     # lon_range=[lon_right, lon_left], lat_range=[lat_bot, lat_up], 
                     lon_range=[-140,-50], lat_range=[15,60],
                     cmap=setcmap, colorbar=False, 
                    grid_line=False, onedecimal=True)
        
        
        # # add title label
        settitle = f'{Timei} Mean'
        # settitle = f'{caseDiffi_label}'
        ax.set_title(f'{settitle}',c='k',fontsize=settitle_size,y=1.02, fontweight='bold'
                    # bbox=dict(facecolor='white', edgecolor='none', boxstyle='round,pad=0.01'),y=0.87,
                    )        
        # If this is the last column, add a colorbar
        # if subploti == 3:
        if subploti == 6:
            norm = plt.Normalize(vmin=0.1, vmax=modified_rangeMax)
            # Create a dummy color-mappable object for the colorbar
            dummy_mappable = ScalarMappable(norm=norm, cmap=setcmap)
            dummy_mappable.set_array([])  # Required for colorbar creation

            # fig.add_axes([left, bottom, width, height])
            cax = fig.add_axes([ax.get_position().x1+0.002, ax.get_position().y0, 0.01, ax.get_position().height*1])
            cbar = fig.colorbar(dummy_mappable, cax=cax, orientation='vertical', extend='both') #extend='both'
            cbar.set_label(setunit, fontsize=setunit_size)
            cbar.ax.tick_params(labelsize=setcolortick_size)
        
        # Hide x and y-axis ticks and labels
        ax.set_xticks([])
        ax.set_yticks([])
        
    elif subploti>6:
        # ax.axis('off')
        # ax.set_axis_off()
        ax.set_visible(False) 
        
axes.flat[-1].set_visible(False)        

# Overall title
plttitle = f'BB Emissions | {spc}'
fig.suptitle(plttitle, fontsize=28, y=1.02, fontweight='bold')

savefig_filename = f'{figure_diri}BBemis_202304T10_CONUS.png'
# Save the figure
plt.savefig(savefig_filename, dpi=300, bbox_inches='tight')  # Adjust the filename and dpi as needed
print(f'Save to {savefig_filename}')
plt.show()

In [ ]:
# Plot emissions monthly means | 2023
lev_idx = -1

Yeari = '2023'
Months_ls = ['04','05','06','07','08','09','10']

# Map setting
setcmap = cm.CMRmap_r
setunit_size, settitle_size, settitle_size2, setcolortick_size = [20, 25, 49, 35]

### Original fire emissions for monthly mean
spc = 'CO'
givenMW = 28 #molecular weight in g/mol
spc_fileINpath = f'{original_diri}qfed.emis_{spc}_bb_surface_daily_20171201T20231231_ne30np4_mol_c20240126.nc'
# Open Xarray dataset
spci_ds = xr.open_dataset(spc_fileINpath)

###-----------------------------------
### Plot a x rows by x columns map
nrows, ncols = 2, 4

# Create a new figure with subplots sharing x and y axes
fig, axes = plt.subplots(
    nrows=nrows, 
    ncols=ncols, 
    figsize=(5.9 * ncols, 3.64 * nrows), 
    sharex=True, 
    sharey=True, 
    subplot_kw={'projection': ccrs.PlateCarree()},  # Set projection for all subplots
    gridspec_kw={'wspace': 0, 'hspace': 0.15}  # Remove space between subplots
)
# fig.tight_layout(pad=0)  # Remove padding

# Quick plot to check
setmap = cm.CMRmap_r 

for subploti in range(len(Months_ls)):
    ax = axes.flat[subploti]
    # ax.set_extent([lon_range[0], lon_range[1], lat_range[0], lat_range[1]], crs=ccrs.PlateCarree())
    if subploti<=6:
        # Get the corresponding subplot
        # Get the data in the corresponding month
        Monthi = Months_ls[subploti]
        Timei = f'{Yeari}T{Monthi}'
        
        """Get the data for the given month (in ne30)"""
        # Convert to integers
        year_int = int(Yeari)
        month_int = int(Monthi)
        # Select the desired month directly using .dt.year and .dt.month
        ds_monthi = spci_ds.sel(time=(spci_ds.time.dt.year == int(Yeari)) & (spci_ds.time.dt.month == int(Monthi)))

        # Compute the monthly mean of the 'emiss' variable
        emiss_monthi_mean = ds_monthi['emiss'].mean(dim='time')

        Plot_ar = emiss_monthi_mean.values#*scalefactor

        vari_kg_per_m2_per_s = molecules_to_kg_per_m2_per_s(Plot_ar, givenMW)
        Plot_unit = r'$kg$ $m^{-2}$ $s^{-1}$'
        longname = spci_ds['emiss'].long_name
        # rangeMax = np.nanmean(Plot_ar) #setvmax*scalefactor
        
        # get rangeMax
        rangeMax = Abs_rangeMax_dic[spc]

        ### Accounting for scientific notation
        # Get the magnitude in scientific notation, adjust for e-
        magnitude = int(np.floor(np.log10(abs(rangeMax))))
        # Format the magnitude as a superscript
        formatted_magnitude = get_superscript(str(magnitude))
        modified_rangeMax = rangeMax*(1/(10)**magnitude)

        setunit = f'{formatted_magnitude}{Plot_unit}'
        
        # show colorbar for the last row in each column 
        im = Plot_2D( vari_kg_per_m2_per_s*(1/(10)**magnitude), scrip_file=SCRIP_ne30,  ax=ax,
                    unit=setunit,unit_size=setunit_size,colortick_size=setcolortick_size,
                    cmin=0.1, cmax=modified_rangeMax,
                     cmap=setcmap, colorbar=False, 
                    grid_line=False, onedecimal=True)
        
        
        # # add title label
        settitle = f'{Timei} Mean'
        # settitle = f'{caseDiffi_label}'
        ax.set_title(f'{settitle}',c='k',fontsize=settitle_size,y=1.02, fontweight='bold'
                    # bbox=dict(facecolor='white', edgecolor='none', boxstyle='round,pad=0.01'),y=0.87,
                    )        
        # If this is the last column, add a colorbar
        # if subploti == 3:
        if subploti == 6:
            norm = plt.Normalize(vmin=0.1, vmax=modified_rangeMax)
            # Create a dummy color-mappable object for the colorbar
            dummy_mappable = ScalarMappable(norm=norm, cmap=setcmap)
            dummy_mappable.set_array([])  # Required for colorbar creation

            # fig.add_axes([left, bottom, width, height])
            cax = fig.add_axes([ax.get_position().x1+0.002, ax.get_position().y0, 0.01, ax.get_position().height*1])
            cbar = fig.colorbar(dummy_mappable, cax=cax, orientation='vertical', extend='both') #extend='both'
            cbar.set_label(setunit, fontsize=setunit_size)
            cbar.ax.tick_params(labelsize=setcolortick_size)
        
        # Hide x and y-axis ticks and labels
        ax.set_xticks([])
        ax.set_yticks([])
        
    elif subploti>6:
        # ax.axis('off')
        # ax.set_axis_off()
        ax.set_visible(False) 
        
axes.flat[-1].set_visible(False)        

# Overall title
plttitle = f'BB Emissions | {spc}'
fig.suptitle(plttitle, fontsize=28, y=1.02, fontweight='bold')

savefig_filename = f'{figure_diri}BBemis_202304T10_Global.png'
# Save the figure
plt.savefig(savefig_filename, dpi=300, bbox_inches='tight')  # Adjust the filename and dpi as needed
print(f'Save to {savefig_filename}')
plt.show()

#### Draft

In [ ]:
# Plot emissions monthly means | 2023
lev_idx = -1

Yeari = '2023'
Months_ls = ['04','05','06','07','08','09','10']

# Map setting
setcmap = cm.CMRmap_r
setunit_size, settitle_size, settitle_size2, setcolortick_size = [20, 25, 49, 35]

### Original fire emissions for monthly mean
spc = 'CO'
givenMW = 28 #molecular weight in g/mol
spc_fileINpath = f'{original_diri}qfed.emis_{spc}_bb_surface_daily_20171201T20231231_ne30np4_mol_c20240126.nc'
# Open Xarray dataset
spci_ds = xr.open_dataset(spc_fileINpath)

###-----------------------------------
### Plot a x rows by x columns map
nrows, ncols = 2, 4

# Create a new figure with subplots sharing x and y axes
fig, axes = plt.subplots(
    nrows=nrows, 
    ncols=ncols, 
    figsize=(5.9 * ncols, 3.64 * nrows), 
    sharex=True, 
    sharey=True, 
    subplot_kw={'projection': ccrs.PlateCarree()},  # Set projection for all subplots
    gridspec_kw={'wspace': 0, 'hspace': 0.25}  # Remove space between subplots
)
# fig.tight_layout(pad=0)  # Remove padding

# Quick plot to check
setmap = cm.CMRmap_r

for subploti in range(len(Months_ls)):
    ax = axes.flat[subploti]
    # ax.set_extent([lon_range[0], lon_range[1], lat_range[0], lat_range[1]], crs=ccrs.PlateCarree())
    if subploti<=6:
        # Get the corresponding subplot
        # Get the data in the corresponding month
        Monthi = Months_ls[subploti]
        Timei = f'{Yeari}T{Monthi}'
        
        """Get the data for the given month (in ne30)"""
        # Convert to integers
        year_int = int(Yeari)
        month_int = int(Monthi)
        # Select the desired month directly using .dt.year and .dt.month
        ds_monthi = spci_ds.sel(time=(spci_ds.time.dt.year == int(Yeari)) & (spci_ds.time.dt.month == int(Monthi)))

        # Compute the monthly mean of the 'emiss' variable
        emiss_monthi_mean = ds_monthi['emiss'].mean(dim='time')

        Plot_ar = emiss_monthi_mean.values#*scalefactor

        vari_kg_per_m2_per_s = molecules_to_kg_per_m2_per_s(Plot_ar, givenMW)
        Plot_unit = r'$kg$ $m^{-2}$ $s^{-1}$'
        longname = spci_ds['emiss'].long_name
        # rangeMax = np.nanmean(Plot_ar) #setvmax*scalefactor
        
        # get rangeMax
        rangeMax = Abs_rangeMax_dic[spc]

        ### Accounting for scientific notation
        # Get the magnitude in scientific notation, adjust for e-
        magnitude = int(np.floor(np.log10(abs(rangeMax))))
        # Format the magnitude as a superscript
        formatted_magnitude = get_superscript(str(magnitude))
        modified_rangeMax = rangeMax*(1/(10)**magnitude)

        setunit = f'{formatted_magnitude}{Plot_unit}'
        
        # show colorbar for the last row in each column 
        im = Plot_2D( vari_kg_per_m2_per_s*(1/(10)**magnitude), scrip_file=SCRIP_ne30,  ax=ax,
                    unit=setunit,unit_size=setunit_size,colortick_size=setcolortick_size,
                    cmin=0.1, cmax=modified_rangeMax,
                     state=True, 
                     # lon_range=[lon_right, lon_left], lat_range=[lat_bot, lat_up], 
                     lon_range=[-140,-50], lat_range=[15,60],
                     cmap=setcmap, colorbar=False, 
                    grid_line=False, onedecimal=True)
        
        
        # # add title label
        settitle = f'CO | BB Emissions \n {Timei} Mean'
        # settitle = f'{caseDiffi_label}'
        ax.set_title(f'{settitle}',c='k',fontsize=settitle_size,y=1.02, fontweight='bold'
                    # bbox=dict(facecolor='white', edgecolor='none', boxstyle='round,pad=0.01'),y=0.87,
                    )        
        # If this is the last column, add a colorbar
        # if subploti == 3:
        if subploti == 6:
            norm = plt.Normalize(vmin=0.1, vmax=modified_rangeMax)
            # Create a dummy color-mappable object for the colorbar
            dummy_mappable = ScalarMappable(norm=norm, cmap=setcmap)
            dummy_mappable.set_array([])  # Required for colorbar creation

            # fig.add_axes([left, bottom, width, height])
            cax = fig.add_axes([ax.get_position().x1+0.002, ax.get_position().y0, 0.01, ax.get_position().height*1])
            cbar = fig.colorbar(dummy_mappable, cax=cax, orientation='vertical', extend='both') #extend='both'
            cbar.set_label(setunit, fontsize=setunit_size)
            cbar.ax.tick_params(labelsize=setcolortick_size)
        
        # Hide x and y-axis ticks and labels
        ax.set_xticks([])
        ax.set_yticks([])
        
    elif subploti>6:
        # ax.axis('off')
        # ax.set_axis_off()
        ax.set_visible(False) 
        
axes.flat[-1].set_visible(False)        



In [ ]:
### Original fire emissions for monthly mean
spc_fileINpath = f'{original_diri}qfed.emis_{spc}_bb_surface_daily_20171201T20231231_ne30np4_mol_c20240126.nc'
# Open Xarray dataset
spci_ds = xr.open_dataset(spc_fileINpath)
givenMW = 28 #molecular weight in g/mol

# Specify the year and month of interest
Yeari = '2022'
Monthi = '07'

# Convert to integers
year_int = int(Yeari)
month_int = int(Monthi)

# Select the desired month directly using .dt.year and .dt.month
ds_monthi = spci_ds.sel(time=(spci_ds.time.dt.year == int(Yeari)) & (spci_ds.time.dt.month == int(Monthi)))

# Compute the monthly mean of the 'emiss' variable
emiss_monthi_mean = ds_monthi['emiss'].mean(dim='time')

Plot_ar = emiss_monthi_mean.values#*scalefactor

### Plotting
# get rangeMax
rangeMax = Abs_rangeMax_dic[spc]

### Accounting for scientific notation
# Get the magnitude in scientific notation, adjust for e-
magnitude = int(np.floor(np.log10(abs(rangeMax))))
# Format the magnitude as a superscript
formatted_magnitude = get_superscript(str(magnitude))
modified_rangeMax = rangeMax*(1/(10)**magnitude)

# Quick plot to check
setmap = cm.CMRmap_r

"""ds_mappedMUSICA (regridded to ne30)"""
PlotRegion = 'CONUS'

# scalefactor = 1e-11
# formatted_label = f"{scalefactor:.0e}" 


vari_kg_per_m2_per_s = molecules_to_kg_per_m2_per_s(Plot_ar, givenMW)
Plot_unit = r'$kg$ $m^{-2}$ $s^{-1}$'
longname = test_ds['emiss'].long_name
rangeMax = np.nanmean(Plot_ar) #setvmax*scalefactor

### Which map
fig = plt.figure( figsize=(8,6) ) 
# - ne30x8 regional refinement over CONUS|
ax1 = fig.add_subplot(1,1,1,projection=ccrs.PlateCarree())
if PlotRegion=="CONUS":
    im = Plot_2D( vari_kg_per_m2_per_s*(1/(10)**magnitude), scrip_file=SCRIP_ne30, ax=ax1,
            cmin=0.1, cmax=modified_rangeMax, cmap=setmap, 
            unit=f'{formatted_magnitude}{Plot_unit}',
            state=True, lon_range=[-140,-50], lat_range=[15,60],
              grid_line=False, grid_line_lw=0.15, twodecimal=True ) 
elif PlotRegion=="Global":
    im = Plot_2D( vari_kg_per_m2_per_s*(1/(10)**magnitude), scrip_file=SCRIP_ne30, ax=ax1,
            cmin=0.1, cmax=modified_rangeMax, cmap=setmap, 
            unit=f'{formatted_magnitude}{Plot_unit}',
            state=True, 
              grid_line=False, grid_line_lw=0.15, twodecimal=True) 

# plt.title(longname+' \n'+Timei[:13], fontsize=16, y=1.02);

In [ ]:
magnitude

In [ ]:
vari_kg_per_m2_per_s*(1/(10)**magnitude)

In [ ]:
modified_rangeMax

In [ ]:
vari_kg_per_m2_per_s*(1/(10)**magnitude)